<img src="https://raw.githubusercontent.com/IDEALLab/EngiOpt/codex/dcc26-workshop-notebooks/workshops/dcc26/assets/engibench_logo.png" width="600"/>

# Notebook 04 Participant - Wrapping a Heat Exchanger Design Problem

**Capstone idea:** take a small, recognizable engineering simulator and wrap it in the same interface you used in the earlier notebooks.

In Notebooks 00-02, `beams2d` was already packaged for you. Here we build a new benchmark-shaped problem from scratch: a compact counterflow heat exchanger.

> Colab users: click **File -> Save a copy in Drive** before editing so your changes persist.


## Where we are in the workshop

The earlier notebooks used an existing EngiBench problem. This notebook flips the direction: instead of asking "how do I train and evaluate a model on a benchmark?", we ask:

**What does a simulator need before it can become a reusable benchmark?**

A heat exchanger is a good capstone because it is not another topology-optimization image. The design is a small vector of geometric choices, the physics is thermal-fluid system performance, and the constraints are things engineers actually care about: heat duty, pressure drop, size, and manufacturability.


## The problem: compact heat exchanger sizing

Imagine a colleague says:

> I have a hot stream and a cold stream. I need at least 5 kW of heat transfer, but I cannot allow more than 35 kPa pressure drop on the cold side. Can an optimizer or generative model propose useful exchanger geometries?

We will use a deliberately small design vector:

| Design variable | Meaning | Typical effect |
|---|---|---|
| `tube_diameter_m` | inner tube diameter | larger diameter lowers pressure drop but can lower velocity and heat transfer coefficient |
| `tube_length_m` | length of each tube | more area, more pressure drop |
| `n_tubes` | number of parallel tubes | more area and lower velocity, but larger/costlier exchanger |

The simulator estimates heat transfer with the effectiveness-NTU method and pressure drop with a Darcy friction-factor model. When `ht` and `fluids` are installed, we use their implementations for those two standard engineering calculations. If not, the notebook falls back to the same textbook formulas so the story still runs.


### Exercise legend

| Marker | Meaning |
|---|---|
| `PUBLIC FILL-IN CELL` | Your turn: edit the code between `START FILL` and `END FILL` |
| `CHECKPOINT` | Automated check: if it fails, fix the fill-in before continuing |


## Install dependencies (Colab / fresh env only)

The notebook is self-contained, but Colab should install `ht` and `fluids` so we demonstrate wrapping real Python engineering libraries.


In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # set True to force install in a local notebook runtime

if IN_COLAB or FORCE_INSTALL:
    def _pip(pkgs):
        subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])

    _pip(["numpy", "pandas", "matplotlib", "scipy", "ht", "fluids"])
    print("Install complete. If imports fail, restart the runtime and rerun from the top.")
else:
    print("Using current environment. Set FORCE_INSTALL=True to install optional libraries here.")


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from types import SimpleNamespace
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from ht.hx import NTU_from_UA, effectiveness_from_NTU
    HT_AVAILABLE = True
except Exception:
    HT_AVAILABLE = False

try:
    from fluids.friction import friction_factor
    FLUIDS_AVAILABLE = True
except Exception:
    FLUIDS_AVAILABLE = False

print("ht available:     ", HT_AVAILABLE)
print("fluids available: ", FLUIDS_AVAILABLE)


---
## 1 - What is fixed by the benchmark?

A reusable design benchmark has to pin down a contract:

1. **Design space:** what the algorithm is allowed to output.
2. **Conditions:** what scenario the design must work under.
3. **Simulator:** how a candidate design is scored.
4. **Constraints:** when a candidate is invalid or suspect.
5. **Baseline optimizer:** a simple reference method to compare against.
6. **Renderer:** a canonical way to inspect the design.

The class below is intentionally small, but it has the same moving parts as an EngiBench `Problem`.


In [ ]:
@dataclass
class Box:
    """Tiny stand-in for a Gymnasium Box design space."""

    low: np.ndarray
    high: np.ndarray
    labels: tuple[str, ...]

    @property
    def shape(self):
        return self.low.shape

    def sample(self, rng):
        return rng.uniform(self.low, self.high).astype(float)

    def clip(self, x):
        return np.clip(np.asarray(x, dtype=float), self.low, self.high)

    def contains(self, x):
        x = np.asarray(x, dtype=float)
        return x.shape == self.shape and np.all(x >= self.low) and np.all(x <= self.high)


@dataclass
class OptiStep:
    obj_values: np.ndarray
    step: int
    design: np.ndarray


---
## 2 - The thermal-fluid model

This is the physics core. We use:

- heat capacity rates: `C = mdot * cp`
- heat-transfer area: `A = pi * D * L * n_tubes`
- overall conductance: `UA = U * A`
- effectiveness-NTU method for counterflow heat exchange
- Darcy-Weisbach pressure drop on the cold side

The design lesson is not that this is the world's most detailed exchanger model. The lesson is that a benchmark should make all assumptions explicit and executable.


In [ ]:
def _fallback_effectiveness_from_ntu(ntu: float, cr: float) -> float:
    """Counterflow heat-exchanger effectiveness from NTU and capacity ratio."""
    ntu = max(float(ntu), 0.0)
    cr = min(max(float(cr), 1e-9), 0.999999)
    if abs(1.0 - cr) < 1e-6:
        return ntu / (1.0 + ntu)
    numerator = 1.0 - math.exp(-ntu * (1.0 - cr))
    denominator = 1.0 - cr * math.exp(-ntu * (1.0 - cr))
    return numerator / denominator


def _fallback_friction_factor(re: float, relative_roughness: float) -> float:
    """Darcy friction factor: laminar exact, turbulent Haaland approximation."""
    re = max(float(re), 1e-9)
    if re < 2300:
        return 64.0 / re
    term = (relative_roughness / 3.7) ** 1.11 + 6.9 / re
    return 1.0 / (-1.8 * math.log10(term)) ** 2


def _ntu_from_ua(ua: float, c_min: float) -> float:
    if HT_AVAILABLE:
        return float(NTU_from_UA(UA=ua, Cmin=c_min))
    return float(ua / max(c_min, 1e-12))


def _effectiveness(ntu: float, cr: float) -> float:
    if HT_AVAILABLE:
        return float(effectiveness_from_NTU(NTU=ntu, Cr=cr, subtype="counterflow"))
    return _fallback_effectiveness_from_ntu(ntu, cr)


def _friction_factor(re: float, eD: float) -> float:
    if FLUIDS_AVAILABLE:
        return float(friction_factor(Re=re, eD=eD, Darcy=True))
    return _fallback_friction_factor(re, eD)


---
## 3 - Wrap the simulator as a problem

Read this class the way you would read a real benchmark implementation. The important question is not "do I like these exact constants?" It is:

**Can another lab run the same design under the same conditions and get the same metrics?**

That is what a benchmark wrapper buys us.


In [ ]:
class HeatExchangerDesignProblem:
    """Small EngiBench-style heat-exchanger design problem."""

    objectives = (
        ("heat_shortfall_W", "MINIMIZE"),
        ("pumping_power_W", "MINIMIZE"),
        ("area_m2", "MINIMIZE"),
    )

    design_space = Box(
        low=np.array([0.006, 0.50, 2.0]),
        high=np.array([0.030, 6.00, 40.0]),
        labels=("tube_diameter_m", "tube_length_m", "n_tubes"),
    )

    default_conditions = {
        "hot_in_C": 80.0,
        "cold_in_C": 20.0,
        "hot_mdot_kg_s": 0.32,
        "cold_mdot_kg_s": 0.24,
        "required_heat_W": 5000.0,
        "max_cold_dp_kPa": 35.0,
        "hot_side_h_W_m2K": 180.0,
    }

    def __init__(self, seed: int = 7, **condition_overrides):
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.conditions = {**self.default_conditions, **condition_overrides}

        # Constant properties keep the notebook focused. A production problem
        # could call CoolProp here for temperature-dependent properties.
        self.cold = SimpleNamespace(rho=997.0, cp=4180.0, mu=1.0e-3, k=0.60, pr=7.0)
        self.hot = SimpleNamespace(rho=850.0, cp=2200.0, mu=3.0e-3, k=0.13, pr=50.0)
        self.wall_k_W_mK = 16.0
        self.wall_thickness_m = 0.001
        self.roughness_m = 1.5e-5
        self.pump_efficiency = 0.65

    def reset(self, seed: int | None = None):
        if seed is not None:
            self.seed = seed
        self.rng = np.random.default_rng(self.seed)

    def unpack_design(self, design):
        d, length, n_tubes = self.design_space.clip(design)
        return float(d), float(length), int(round(float(n_tubes)))

    def _inside_heat_transfer_coefficient(self, diameter_m, n_tubes, cold_mdot):
        area_per_tube = math.pi * diameter_m**2 / 4.0
        velocity = cold_mdot / (self.cold.rho * area_per_tube * max(n_tubes, 1))
        reynolds = self.cold.rho * velocity * diameter_m / self.cold.mu
        if reynolds < 2300:
            nusselt = 3.66
        else:
            nusselt = 0.023 * reynolds**0.8 * self.cold.pr**0.4
        h_inside = nusselt * self.cold.k / diameter_m
        return h_inside, velocity, reynolds

    def simulate(self, design, config: dict | None = None) -> np.ndarray:
        cfg = {**self.conditions, **(config or {})}
        diameter_m, length_m, n_tubes = self.unpack_design(design)

        h_inside, velocity, reynolds = self._inside_heat_transfer_coefficient(
            diameter_m, n_tubes, cfg["cold_mdot_kg_s"]
        )
        h_outside = float(cfg["hot_side_h_W_m2K"])
        u_overall = 1.0 / (
            1.0 / h_inside
            + self.wall_thickness_m / self.wall_k_W_mK
            + 1.0 / h_outside
        )

        area_m2 = math.pi * diameter_m * length_m * n_tubes
        ua = u_overall * area_m2

        c_hot = cfg["hot_mdot_kg_s"] * self.hot.cp
        c_cold = cfg["cold_mdot_kg_s"] * self.cold.cp
        c_min = min(c_hot, c_cold)
        c_max = max(c_hot, c_cold)
        cr = c_min / c_max
        ntu = _ntu_from_ua(ua, c_min)
        eps = _effectiveness(ntu, cr)

        q_max = c_min * (cfg["hot_in_C"] - cfg["cold_in_C"])
        q_W = eps * q_max
        heat_shortfall_W = max(cfg["required_heat_W"] - q_W, 0.0)

        eD = self.roughness_m / diameter_m
        f_darcy = _friction_factor(reynolds, eD)
        minor_loss_K = 1.5
        cold_dp_Pa = (f_darcy * length_m / diameter_m + minor_loss_K) * 0.5 * self.cold.rho * velocity**2
        pumping_power_W = cold_dp_Pa * (cfg["cold_mdot_kg_s"] / self.cold.rho) / self.pump_efficiency

        cold_out_C = cfg["cold_in_C"] + q_W / c_cold
        hot_out_C = cfg["hot_in_C"] - q_W / c_hot

        self.last_details = {
            "diameter_m": diameter_m,
            "length_m": length_m,
            "n_tubes": n_tubes,
            "area_m2": area_m2,
            "U_W_m2K": u_overall,
            "UA_W_K": ua,
            "NTU": ntu,
            "effectiveness": eps,
            "heat_transfer_W": q_W,
            "heat_shortfall_W": heat_shortfall_W,
            "cold_dp_kPa": cold_dp_Pa / 1000.0,
            "pumping_power_W": pumping_power_W,
            "cold_velocity_m_s": velocity,
            "cold_reynolds": reynolds,
            "cold_out_C": cold_out_C,
            "hot_out_C": hot_out_C,
        }

        return np.array([heat_shortfall_W, pumping_power_W, area_m2], dtype=float)

    def check_constraints(self, design, config: dict | None = None) -> list[str]:
        cfg = {**self.conditions, **(config or {})}
        violations = []
        x = np.asarray(design, dtype=float)
        if not self.design_space.contains(x):
            violations.append("design is outside geometry bounds")

        self.simulate(x, cfg)
        d = self.last_details
        if d["heat_shortfall_W"] > 1e-6:
            violations.append("required heat duty is not met")
        if d["cold_dp_kPa"] > cfg["max_cold_dp_kPa"]:
            violations.append("cold-side pressure drop exceeds limit")
        if d["cold_velocity_m_s"] < 0.20:
            violations.append("cold-side velocity is very low; fouling risk")
        if d["cold_velocity_m_s"] > 3.00:
            violations.append("cold-side velocity is high; erosion/noise risk")
        min_approach_C = 2.0
        if cfg["hot_in_C"] - d["cold_out_C"] < min_approach_C:
            violations.append("hot-in to cold-out terminal approach is too small")
        if d["hot_out_C"] - cfg["cold_in_C"] < min_approach_C:
            violations.append("hot-out to cold-in terminal approach is too small")
        return violations

    def random_design(self):
        return self.design_space.sample(self.rng), -1.0

    def score(self, design, config: dict | None = None) -> float:
        cfg = {**self.conditions, **(config or {})}
        obj = self.simulate(design, cfg)
        d = self.last_details
        pressure_penalty = max(d["cold_dp_kPa"] - cfg["max_cold_dp_kPa"], 0.0) / cfg["max_cold_dp_kPa"]
        velocity_penalty = max(0.20 - d["cold_velocity_m_s"], 0.0) + max(d["cold_velocity_m_s"] - 3.00, 0.0)
        return (
            obj[0] / cfg["required_heat_W"]
            + 0.02 * obj[1]
            + 0.08 * obj[2]
            + 10.0 * pressure_penalty
            + 2.0 * velocity_penalty
        )

    def optimize(self, starting_point=None, config: dict | None = None, n_candidates: int = 600):
        if starting_point is None:
            starting_point, _ = self.random_design()
        best = self.design_space.clip(starting_point)
        best_score = self.score(best, config)
        best_obj = self.simulate(best, config)
        history = [OptiStep(obj_values=best_obj, step=0, design=best.copy())]

        for step in range(1, n_candidates + 1):
            candidate = self.design_space.sample(self.rng)
            candidate_score = self.score(candidate, config)
            if candidate_score < best_score:
                best = candidate.copy()
                best_score = candidate_score
                best_obj = self.simulate(best, config)
            if step % 20 == 0:
                history.append(OptiStep(obj_values=best_obj.copy(), step=step, design=best.copy()))
        return best, history

    def render(self, design, config: dict | None = None):
        self.simulate(design, config)
        d = self.last_details
        labels = self.design_space.labels
        values = self.design_space.clip(design)

        fig, axes = plt.subplots(1, 3, figsize=(14, 4))

        # Schematic panel
        ax = axes[0]
        ax.set_title("Counterflow tube bundle")
        ax.plot([0.08, 0.92], [0.65, 0.65], color="#b91c1c", linewidth=6, solid_capstyle="round")
        ax.plot([0.92, 0.08], [0.35, 0.35], color="#2563eb", linewidth=6, solid_capstyle="round")
        for y in np.linspace(0.40, 0.60, 5):
            ax.plot([0.16, 0.84], [y, y], color="0.25", linewidth=1.5, alpha=0.8)
        ax.text(0.08, 0.74, f"hot in {self.conditions['hot_in_C']:.0f} C", color="#b91c1c")
        ax.text(0.70, 0.24, f"cold in {self.conditions['cold_in_C']:.0f} C", color="#2563eb")
        ax.text(0.08, 0.08, f"D={d['diameter_m']*1000:.1f} mm, L={d['length_m']:.2f} m, tubes={d['n_tubes']}")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")

        # Objective panel
        ax = axes[1]
        names = ["Q delivered", "Q required"]
        vals = [d["heat_transfer_W"] / 1000, self.conditions["required_heat_W"] / 1000]
        ax.bar(names, vals, color=["#0f766e", "#525252"])
        ax.set_ylabel("kW")
        ax.set_title("Heat duty")
        ax.grid(axis="y", alpha=0.25)

        # Constraint / tradeoff panel
        ax = axes[2]
        names = ["dp", "limit", "area", "pump"]
        vals = [d["cold_dp_kPa"], self.conditions["max_cold_dp_kPa"], d["area_m2"], d["pumping_power_W"]]
        colors = ["#7c3aed", "#525252", "#ea580c", "#0891b2"]
        ax.bar(names, vals, color=colors)
        ax.set_title("Pressure, size, power")
        ax.set_ylabel("mixed units")
        ax.grid(axis="y", alpha=0.25)

        fig.suptitle("HeatExchangerDesignProblem.render(design)", fontsize=14)
        fig.tight_layout()
        return fig


---
## 4 - Instantiate and inspect the problem

This is the moment where a simulator starts to feel like a benchmark: we can inspect the design space, the objectives, and the operating scenario before running any optimization.


In [ ]:
problem = HeatExchangerDesignProblem(seed=4)

print("Design variables:")
for label, lo, hi in zip(problem.design_space.labels, problem.design_space.low, problem.design_space.high):
    print(f"  {label:18s}: {lo:.4g} to {hi:.4g}")

print("\nObjectives:")
for name, direction in problem.objectives:
    print(f"  {name:18s}: {direction}")

print("\nConditions:")
for k, v in problem.conditions.items():
    print(f"  {k:18s}: {v}")


---
## 5 - FILL-IN 04-A: Simulate one candidate

A benchmark is more than a dataset. It should let us ask: *if a model gives me this design, what happens under this scenario?*

Your first task is to choose a candidate heat-exchanger geometry. Bigger is not always better: longer tubes and more tubes add area, but pressure drop and pumping power can fight back.

Use the design-space bounds printed above:

1. `tube_diameter_m` between 0.006 and 0.030
2. `tube_length_m` between 0.50 and 6.00
3. `n_tubes` between 2 and 40


In [ ]:
# PUBLIC FILL-IN CELL 04-A
# Goal: choose one candidate heat-exchanger geometry and simulate it.

# START FILL ---------------------------------------------------------------
# Replace None with a 3-value numpy array:
#   [tube_diameter_m, tube_length_m, n_tubes]
# Example shape only: np.array([0.014, 3.20, 14.0])
candidate = None
# END FILL -----------------------------------------------------------------

if candidate is None:
    raise RuntimeError("Set candidate to a 3-value numpy array before continuing.")

candidate = np.asarray(candidate, dtype=float)

# CHECKPOINT
assert candidate.shape == problem.design_space.shape, f"Expected shape {problem.design_space.shape}, got {candidate.shape}"
assert problem.design_space.contains(candidate), "Candidate is outside the design-space bounds."

obj = problem.simulate(candidate)
violations = problem.check_constraints(candidate)

# CHECKPOINT
assert np.all(np.isfinite(obj)), "Simulation produced non-finite objective values."
assert "heat_transfer_W" in problem.last_details, "Simulator details were not recorded."

print("Objective vector [heat_shortfall_W, pumping_power_W, area_m2]:")
print(np.round(obj, 4))

print("\nDetails:")
for key in ["heat_transfer_W", "cold_dp_kPa", "pumping_power_W", "area_m2", "effectiveness", "cold_reynolds", "cold_velocity_m_s", "hot_out_C", "cold_out_C"]:
    print(f"  {key:18s}: {problem.last_details[key]:.4g}")

print("\nConstraint violations:")
print(violations if violations else "  none")
print("\nCHECKPOINT passed - candidate simulated successfully.")


In [ ]:
fig = problem.render(candidate)
plt.show()


---
## 6 - Why constraints are separate from objectives

A design can have a small area and low pumping power because it simply fails to transfer enough heat. That is why `simulate()` and `check_constraints()` answer different questions.

- `simulate()` says how the design performs.
- `check_constraints()` says whether the performance is acceptable for this benchmark scenario.

This separation is exactly what made Notebook 02 useful: a design can look plausible and still fail engineering checks.


In [ ]:
examples = {
    "too small": np.array([0.010, 0.70, 4.0]),
    "pressure-heavy": np.array([0.0065, 5.50, 3.0]),
    "reasonable": candidate,
}

rows = []
for name, x in examples.items():
    obj = problem.simulate(x)
    rows.append({
        "case": name,
        "D_mm": problem.last_details["diameter_m"] * 1000,
        "L_m": problem.last_details["length_m"],
        "n_tubes": problem.last_details["n_tubes"],
        "Q_kW": problem.last_details["heat_transfer_W"] / 1000,
        "shortfall_W": obj[0],
        "dp_kPa": problem.last_details["cold_dp_kPa"],
        "pump_W": obj[1],
        "area_m2": obj[2],
        "violations": "; ".join(problem.check_constraints(x)) or "none",
    })

pd.DataFrame(rows)


---
## 7 - FILL-IN 04-B: Run a tiny baseline optimizer

For a real EngiBench contribution, the optimizer should be documented carefully: what it optimizes, how long it runs, and whether it is meant to be strong or just a baseline.

Here we use random search because it is transparent. The point is not that random search is clever. The point is that every benchmark needs a reference method that everyone can rerun.


In [ ]:
# PUBLIC FILL-IN CELL 04-B
# Goal: run the baseline optimizer from a random starting design.

problem.reset(seed=12)
start, _ = problem.random_design()

# START FILL ---------------------------------------------------------------
# Call problem.optimize(...) using the random start above.
# Hint: use starting_point=start and n_candidates=800.
best_design = None
history = None
# END FILL -----------------------------------------------------------------

if best_design is None or history is None:
    raise RuntimeError("Call problem.optimize(...) and assign best_design, history.")

# CHECKPOINT
assert len(history) > 1, "Optimization history should contain multiple recorded steps."
assert problem.design_space.contains(best_design), "Best design is outside the design-space bounds."

print("Start design:", dict(zip(problem.design_space.labels, np.round(start, 4))))
problem.simulate(start)
print("Start details:", {k: round(problem.last_details[k], 4) for k in ["heat_transfer_W", "cold_dp_kPa", "pumping_power_W", "area_m2"]})
print("Start violations:", problem.check_constraints(start) or "none")

print("\nBest design:", dict(zip(problem.design_space.labels, np.round(best_design, 4))))
problem.simulate(best_design)
print("Best details:", {k: round(problem.last_details[k], 4) for k in ["heat_transfer_W", "cold_dp_kPa", "pumping_power_W", "area_m2"]})
print("Best violations:", problem.check_constraints(best_design) or "none")

# CHECKPOINT
assert problem.last_details["heat_transfer_W"] >= 0.95 * problem.conditions["required_heat_W"], (
    "The best design is still far below the heat-duty target. Try increasing n_candidates."
)
print("\nCHECKPOINT passed - baseline optimizer produced a useful candidate.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

steps = [h.step for h in history]
shortfall = [h.obj_values[0] for h in history]
pump = [h.obj_values[1] for h in history]
area = [h.obj_values[2] for h in history]

axes[0].plot(steps, shortfall, marker="o", label="heat shortfall [W]")
axes[0].set_xlabel("candidate evaluations")
axes[0].set_ylabel("W")
axes[0].set_title("Best heat-duty shortfall so far")
axes[0].grid(alpha=0.25)

axes[1].plot(steps, pump, marker="o", label="pumping power [W]")
axes[1].plot(steps, area, marker="s", label="area [m2]")
axes[1].set_xlabel("candidate evaluations")
axes[1].set_title("Competing costs of the best design")
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()


In [ ]:
fig = problem.render(best_design)
plt.show()


---
## 8 - FILL-IN 04-C: Change the operating scenario

Conditions are the input side of the benchmark. The same design can be good for one scenario and bad for another.

This is what makes conditional design interesting: an inverse model should not just produce "a heat exchanger". It should produce a heat exchanger for *this* duty, *these* flow rates, and *this* pressure-drop limit.

Your task is to add one new scenario to the list. Keep the keys the same as the existing entries.


In [ ]:
# PUBLIC FILL-IN CELL 04-C
# Goal: add one operating condition, then optimize a design for each scenario.

scenarios = [
    {"name": "base", "required_heat_W": 5000.0, "cold_mdot_kg_s": 0.24, "max_cold_dp_kPa": 35.0},
    {"name": "harder duty", "required_heat_W": 7000.0, "cold_mdot_kg_s": 0.24, "max_cold_dp_kPa": 35.0},
    {"name": "tight dp", "required_heat_W": 5000.0, "cold_mdot_kg_s": 0.24, "max_cold_dp_kPa": 15.0},
    {"name": "more flow", "required_heat_W": 5000.0, "cold_mdot_kg_s": 0.40, "max_cold_dp_kPa": 35.0},
]

# START FILL ---------------------------------------------------------------
# Add one more scenario dict to scenarios.
# Hint: choose a new name and change required_heat_W, cold_mdot_kg_s, or max_cold_dp_kPa.
# Example shape only:
# scenarios.append({"name": "your case", "required_heat_W": 6000.0, "cold_mdot_kg_s": 0.30, "max_cold_dp_kPa": 25.0})
# END FILL -----------------------------------------------------------------

# CHECKPOINT
assert len(scenarios) >= 5, "Add at least one scenario to the list."
required_keys = {"name", "required_heat_W", "cold_mdot_kg_s", "max_cold_dp_kPa"}
for scenario in scenarios:
    assert required_keys <= set(scenario), f"Scenario is missing keys: {scenario}"

rows = []
for scenario in scenarios:
    cfg = {k: v for k, v in scenario.items() if k != "name"}
    problem.reset(seed=30)
    best, _ = problem.optimize(config=cfg, n_candidates=700)
    obj = problem.simulate(best, cfg)
    rows.append({
        "scenario": scenario["name"],
        "D_mm": problem.last_details["diameter_m"] * 1000,
        "L_m": problem.last_details["length_m"],
        "n_tubes": problem.last_details["n_tubes"],
        "Q_kW": problem.last_details["heat_transfer_W"] / 1000,
        "required_kW": cfg["required_heat_W"] / 1000,
        "dp_kPa": problem.last_details["cold_dp_kPa"],
        "dp_limit_kPa": cfg["max_cold_dp_kPa"],
        "area_m2": obj[2],
        "violations": "; ".join(problem.check_constraints(best, cfg)) or "none",
    })

scenario_df = pd.DataFrame(rows)
print("CHECKPOINT passed - scenario sweep complete.")
scenario_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(scenario_df))
ax.bar(x - 0.2, scenario_df["Q_kW"], width=0.4, label="delivered")
ax.bar(x + 0.2, scenario_df["required_kW"], width=0.4, label="required")
ax.set_xticks(x)
ax.set_xticklabels(scenario_df["scenario"], rotation=15, ha="right")
ax.set_ylabel("heat duty [kW]")
ax.set_title("Different conditions lead to different designs")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()


---
## 9 - What would make this a real EngiBench problem?

This notebook is a workshop wrapper, not a polished repository contribution. To turn it into a real EngiBench problem, we would still need:

1. A module under `engibench/problems/heat_exchanger/` with a `v0.py` implementation.
2. A documented dataset of optimized designs across sampled conditions.
3. Tests that every design in the dataset simulates and passes constraints.
4. A stronger baseline optimizer and fixed evaluation budget.
5. Clear citations for the heat-transfer and pressure-drop correlations.
6. Documentation and a canonical render image.

The important thing is that the shape is now visible. Once the simulator is wrapped, any model in EngiOpt can treat this like another conditional design problem.


## Reflection

Before closing the capstone, discuss:

1. Which parts of this problem are **conditions** and which are **design variables**?
2. Is pressure drop an objective, a constraint, or both? What changes if you move it?
3. What data would you generate before training a conditional design model?
4. Which simplification in this notebook would matter most for a publication-grade benchmark?
5. How is this different from the heat-conduction topology problems already in EngiBench?
